# Site Filtering: High Flood Risk & Data Quality

Goal: Filter Missouri Basin sites to keep only those with:
1. High streamflow variation (flood risk)
2. Good data quality (low null rates)

This analysis determines which sites to keep in the main `flood_model` table.

In [14]:
import polars as pl
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import wandb
import numpy as np

In [15]:
# Load the flood model dataset from wandb
api = wandb.Api()
artifact = api.artifact("flood-forecasting/flood-dataset:latest")
artifact_dir = artifact.download()

df = pl.read_parquet(f"{artifact_dir}/flood_model.parquet")
print(f"Full dataset: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Total sites: {df['site_id'].n_unique()}")
print(f"Date range: {df['observation_hour'].min()} to {df['observation_hour'].max()}")

wandb: Downloading large artifact 'flood-dataset:latest', 4522.32MB. 1 files...
wandb:   1 of 1 files downloaded.  
Done. 00:00:00.2 (20649.9MB/s)


Full dataset: 101,651,130 rows x 51 columns
Total sites: 1029
Date range: 2007-10-28 05:00:00+00:00 to 2026-02-03 18:00:00+00:00


## 1. Analyze Streamflow Variation Per Site

We compute the coefficient of variation (CV) of streamflow for each site.
CV = std / mean. Higher CV means more variable flow, i.e., higher flood risk.

In [16]:
# Clean bad values: sentinels (-999999), polluted averages, and impossible negatives
df = df.with_columns(
    pl.when(pl.col("streamflow_cfs_mean") < 0).then(None).otherwise(pl.col("streamflow_cfs_mean")).alias("streamflow_cfs_mean"),
    pl.when(pl.col("gage_height_ft_mean") < -100).then(None).otherwise(pl.col("gage_height_ft_mean")).alias("gage_height_ft_mean"),
)
print(f"Cleaned bad values: streamflow < 0 → null, gage height < -100 → null")

# Compute streamflow statistics per site
site_stats = df.group_by("site_id").agg(
    pl.col("streamflow_cfs_mean").mean().alias("streamflow_mean"),
    pl.col("streamflow_cfs_mean").std().alias("streamflow_std"),
    pl.col("streamflow_cfs_mean").max().alias("streamflow_max"),
    pl.col("streamflow_cfs_mean").min().alias("streamflow_min"),
    pl.col("gage_height_ft_mean").mean().alias("gage_height_mean"),
    pl.col("gage_height_ft_mean").std().alias("gage_height_std"),
    pl.col("latitude").first(),
    pl.col("longitude").first(),
    pl.col("station_name").first(),
    pl.len().alias("total_rows"),
    pl.col("streamflow_cfs_mean").null_count().alias("streamflow_nulls"),
    pl.col("gage_height_ft_mean").null_count().alias("gage_height_nulls"),
    pl.col("precipitation_mm").null_count().alias("precip_nulls"),
    pl.col("temperature_c").null_count().alias("temp_nulls"),
)

# Compute coefficient of variation and null rates
site_stats = site_stats.with_columns(
    (pl.col("streamflow_std") / pl.col("streamflow_mean")).alias("streamflow_cv"),
    (pl.col("streamflow_max") - pl.col("streamflow_min")).alias("streamflow_range"),
    (pl.col("streamflow_nulls") / pl.col("total_rows") * 100).alias("streamflow_null_pct"),
    (pl.col("gage_height_nulls") / pl.col("total_rows") * 100).alias("gage_height_null_pct"),
    (pl.col("precip_nulls") / pl.col("total_rows") * 100).alias("precip_null_pct"),
    (pl.col("temp_nulls") / pl.col("total_rows") * 100).alias("temp_null_pct"),
)

print(f"Total sites: {len(site_stats)}")
print(f"Sites with streamflow data (non-null CV): {site_stats.filter(pl.col('streamflow_cv').is_not_null()).shape[0]}")
print(f"Sites with 100% null streamflow (no data): {site_stats.filter(pl.col('streamflow_null_pct') == 100).shape[0]}")
print(f"Sites with zero mean streamflow: {site_stats.filter(pl.col('streamflow_mean') == 0).shape[0]}")

# Remove sites with no data or zero mean (zero mean causes inf CV)
site_stats = site_stats.filter(
    pl.col("streamflow_cv").is_not_null()
    & pl.col("streamflow_cv").is_finite()
    & (pl.col("streamflow_mean") > 0)
)
print(f"Sites after removing no-data and zero-mean: {len(site_stats)}")

# Show top 20 sites by CV
print("\nTop 20 sites by streamflow variation (highest flood risk):")
site_stats.select(
    "site_id", "station_name", "streamflow_cv", "streamflow_mean",
    "streamflow_range", "total_rows", "streamflow_null_pct", "gage_height_null_pct"
).sort("streamflow_cv", descending=True).head(20)

Replaced sentinel values (-999999) with null
Total sites: 1029
Sites with streamflow data (non-null CV): 939
Sites with 100% null streamflow (no data): 90
Sites with zero mean streamflow: 2
Sites after removing no-data and zero-mean: 921

Top 20 sites by streamflow variation (highest flood risk):


site_id,station_name,streamflow_cv,streamflow_mean,streamflow_range,total_rows,streamflow_null_pct,gage_height_null_pct
str,str,f64,f64,f64,u32,f64,f64
"""06746110""","""JOE WRIGHT CREEK BELOW JOE WRI…",795.383079,4.468853,750293.8125,99089,0.135232,71.094672
"""06862700""","""SMOKY HILL R NR SCHOENCHEN, KS""",355.924981,5.78948,754098.6625,145403,8.784551,0.109351
"""06893557""","""Brush Creek at Ward Parkway in…",279.03071,6.32514,676865.78,143634,0.332094,0.285448
"""06746095""","""JOE WRIGHT CREEK ABOVE JOE WRI…",259.928538,10.383658,666882.436667,61063,0.067144,81.225947
"""06901205""","""East Locust Creek near Boynton…",232.32854,13.3591,759286.03,91890,0.765045,1.225378
…,…,…,…,…,…,…,…
"""06930015""","""McCourtney Hollow Trib at FLW(…",41.26826,0.04586,142.125,15428,0.0,0.155561
"""06741510""","""BIG THOMPSON RIVER AT LOVELAND…",41.030237,58.702688,752218.65,141084,0.254458,77.339032
"""06318500""","""CLEAR CREEK NEAR BUFFALO, WY""",40.656846,65.7055,751697.715,82538,4.268337,31.694492


In [17]:
# Distribution of streamflow CV
cv_data = site_stats.to_pandas()
median_cv = cv_data["streamflow_cv"].median()

# Remove extreme outliers from plot data so bins spread evenly
p95 = cv_data["streamflow_cv"].quantile(0.95)
cv_clipped = cv_data[cv_data["streamflow_cv"] <= p95].copy()
n_outliers = len(cv_data) - len(cv_clipped)

fig = make_subplots(rows=2, cols=1, row_heights=[0.7, 0.3],
    subplot_titles=(
        f"Histogram of Streamflow CV ({len(cv_data)} sites, {n_outliers} outliers clipped)",
        "Box Plot (full range, shows outliers)"
    ))

fig.add_trace(
    go.Histogram(x=cv_clipped["streamflow_cv"], nbinsx=30,
                 name="Sites", marker_color="steelblue"),
    row=1, col=1
)
fig.add_vline(x=median_cv, line_dash="dash", line_color="red",
              annotation_text=f"Median: {median_cv:.2f}", row=1, col=1)

fig.add_trace(
    go.Box(x=cv_data["streamflow_cv"], name="CV", marker_color="steelblue"),
    row=2, col=1
)

fig.update_layout(height=500, showlegend=False, bargap=0.05)
fig.update_xaxes(title_text="CV (std/mean)", row=1, col=1)
fig.update_yaxes(title_text="Number of Sites", row=1, col=1)
fig.show()

print(f"CV range: {cv_data['streamflow_cv'].min():.2f} to {cv_data['streamflow_cv'].max():.2f}")
print(f"Median CV: {median_cv:.2f}")
print(f"Sites above median: {len(cv_data[cv_data['streamflow_cv'] > median_cv])}")
print(f"Sites below median: {len(cv_data[cv_data['streamflow_cv'] <= median_cv])}")
print(f"Outliers clipped from histogram (CV > {p95:.2f}): {n_outliers}")

CV range: 0.06 to 795.38
Median CV: 2.03
Sites above median: 460
Sites below median: 461
Outliers clipped from histogram (CV > 11.40): 46


## 2. Analyze Null Rates Per Site

In [18]:
# Distribution of null rates
null_data = site_stats.to_pandas()

fig = make_subplots(rows=1, cols=2,
    subplot_titles=("Streamflow Null %", "Gage Height Null %"))

fig.add_trace(
    go.Histogram(x=null_data["streamflow_null_pct"], nbinsx=50, name="Streamflow"),
    row=1, col=1
)
fig.add_trace(
    go.Histogram(x=null_data["gage_height_null_pct"], nbinsx=50, name="Gage Height"),
    row=1, col=2
)
fig.update_layout(title="Distribution of Null Rates Across Sites", showlegend=False)
fig.show()

# Summary
print(f"Sites with 0% streamflow nulls:    {len(null_data[null_data['streamflow_null_pct'] == 0])}")
print(f"Sites with <20% streamflow nulls:  {len(null_data[null_data['streamflow_null_pct'] < 20])}")
print(f"Sites with >50% streamflow nulls:  {len(null_data[null_data['streamflow_null_pct'] > 50])}")
print(f"Sites with 100% streamflow nulls:  {len(null_data[null_data['streamflow_null_pct'] == 100])}")
print()
print(f"Sites with 0% gage height nulls:   {len(null_data[null_data['gage_height_null_pct'] == 0])}")
print(f"Sites with <20% gage height nulls: {len(null_data[null_data['gage_height_null_pct'] < 20])}")
print(f"Sites with >50% gage height nulls: {len(null_data[null_data['gage_height_null_pct'] > 50])}")
print(f"Sites with 100% gage height nulls: {len(null_data[null_data['gage_height_null_pct'] == 100])}")

Sites with 0% streamflow nulls:    150
Sites with <20% streamflow nulls:  796
Sites with >50% streamflow nulls:  7
Sites with 100% streamflow nulls:  0

Sites with 0% gage height nulls:   28
Sites with <20% gage height nulls: 605
Sites with >50% gage height nulls: 284
Sites with 100% gage height nulls: 139


## 3. Apply Filters

Criteria:
- Streamflow CV > median (above-average variation = higher flood risk)
- Gage height null rate < 20% (good data quality)
- Streamflow null rate < 20% (good data quality)

Adjust thresholds based on the distributions above.

In [19]:
# Set thresholds (adjust after reviewing distributions above)
CV_THRESHOLD = site_stats.filter(
    pl.col("streamflow_cv").is_not_null()
)["streamflow_cv"].median()
NULL_THRESHOLD = 20  # max % nulls allowed

print(f"CV threshold (median): {CV_THRESHOLD:.2f}")
print(f"Null threshold: {NULL_THRESHOLD}%")

# Apply filters
filtered_sites = site_stats.filter(
    (pl.col("streamflow_cv") > CV_THRESHOLD)
    & (pl.col("streamflow_null_pct") < NULL_THRESHOLD)
    & (pl.col("gage_height_null_pct") < NULL_THRESHOLD)
)

print(f"\nBefore filtering: {len(site_stats)} sites")
print(f"After filtering:  {len(filtered_sites)} sites")
print(f"Removed:          {len(site_stats) - len(filtered_sites)} sites")

# Breakdown of why sites were removed
no_data = site_stats.filter(pl.col("streamflow_cv").is_null())
low_cv = site_stats.filter(
    (pl.col("streamflow_cv").is_not_null()) & (pl.col("streamflow_cv") <= CV_THRESHOLD)
)
high_nulls = site_stats.filter(
    (pl.col("streamflow_cv") > CV_THRESHOLD)
    & ((pl.col("streamflow_null_pct") >= NULL_THRESHOLD) | (pl.col("gage_height_null_pct") >= NULL_THRESHOLD))
)
print(f"\nRemoval breakdown:")
print(f"  No streamflow data at all: {len(no_data)}")
print(f"  Low variation (CV <= {CV_THRESHOLD:.2f}): {len(low_cv)}")
print(f"  High nulls (>= {NULL_THRESHOLD}%): {len(high_nulls)}")

CV threshold (median): 2.03
Null threshold: 20%

Before filtering: 921 sites
After filtering:  268 sites
Removed:          653 sites

Removal breakdown:
  No streamflow data at all: 0
  Low variation (CV <= 2.03): 461
  High nulls (>= 20%): 192


## 4. Visualizations: Before vs After

In [20]:
# Map: All sites vs Filtered sites
all_sites_pd = site_stats.to_pandas()
all_sites_pd["status"] = "Removed"
kept_ids = set(filtered_sites["site_id"].to_list())
all_sites_pd.loc[all_sites_pd["site_id"].isin(kept_ids), "status"] = "Kept"

fig = px.scatter_geo(
    all_sites_pd,
    lat="latitude", lon="longitude",
    color="status",
    color_discrete_map={"Kept": "blue", "Removed": "red"},
    hover_name="station_name",
    hover_data=["site_id", "streamflow_cv", "streamflow_null_pct", "gage_height_null_pct"],
    title=f"Site Filtering: {len(filtered_sites)} Kept (blue) vs {len(site_stats) - len(filtered_sites)} Removed (red)",
    scope="usa",
)
fig.update_layout(geo=dict(center=dict(lat=45, lon=-105), projection_scale=3))
fig.show()

In [21]:
# Scatter: CV vs Null Rate (shows filtering logic)
# Only show sites that have streamflow data (non-null CV)
scatter_data = all_sites_pd.copy()

fig = px.scatter(
    scatter_data,
    x="streamflow_cv", y="gage_height_null_pct",
    color="status",
    color_discrete_map={"Kept": "blue", "Removed": "red"},
    hover_name="station_name",
    hover_data=["site_id"],
    title="Streamflow Variation vs Data Quality (sites with data only)",
    labels={"streamflow_cv": "Streamflow CV (higher = more variable)",
            "gage_height_null_pct": "Gage Height Null %"},
)
fig.add_vline(x=CV_THRESHOLD, line_dash="dash", line_color="gray",
              annotation_text=f"CV threshold: {CV_THRESHOLD:.2f}")
fig.add_hline(y=NULL_THRESHOLD, line_dash="dash", line_color="gray",
              annotation_text=f"Null threshold: {NULL_THRESHOLD}%")
fig.show()

In [22]:
# Comparison: dataset size before vs after
kept_df = df.filter(pl.col("site_id").is_in(filtered_sites["site_id"].to_list()))

print(f"Rows before: {len(df):,}")
print(f"Rows after:  {len(kept_df):,}")
print(f"Reduction:   {(1 - len(kept_df)/len(df))*100:.1f}%")
print(f"\nSites before: {df['site_id'].n_unique()}")
print(f"Sites after:  {kept_df['site_id'].n_unique()}")

Rows before: 101,651,130
Rows after:  30,741,717
Reduction:   69.8%

Sites before: 1029
Sites after:  268


In [23]:
# Export the list of kept site IDs as a dbt seed CSV
from pathlib import Path

kept_site_ids = filtered_sites.select("site_id").sort("site_id")
print(f"Kept {len(kept_site_ids)} site IDs")

seed_path = Path("../elt/transformation/seeds/filtered_site_ids.csv")
kept_site_ids.write_csv(seed_path)
print(f"Saved to {seed_path.resolve()}")
print(kept_site_ids)

Kept 268 site IDs
Saved to C:\Users\sacha\RiceLocal\Capstone\Coding\Flood-Forecasting\elt\transformation\seeds\filtered_site_ids.csv
shape: (268, 1)
┌─────────────────┐
│ site_id         │
│ ---             │
│ str             │
╞═════════════════╡
│ 06033000        │
│ 06048650        │
│ 06052500        │
│ 06061500        │
│ 06062500        │
│ …               │
│ 06935980        │
│ 06935997        │
│ 06936475        │
│ 06936530        │
│ 411450095582201 │
└─────────────────┘


In [24]:
# First few rows
kept_df.head(10)

site_id,observation_hour,latitude,longitude,streamflow_cfs_mean,streamflow_cfs_max,streamflow_cfs_min,gage_height_ft_mean,gage_height_ft_max,gage_height_ft_min,observation_count,precipitation_mm,temperature_c,wind_speed_ms,specific_humidity_kgkg,surface_pressure_pa,shortwave_radiation_wm2,longwave_radiation_wm2,potential_evaporation_mm,cape_jkg,convective_precip_fraction,station_name,huc_code,drainage_area_sq_km,is_reference_hcdn2009,elev_mean_m,elev_max_m,elev_min_m,SLOPE_PCT,ASPECT_NORTHNESS,ASPECT_EASTNESS,geology_class_reedbush,geology_desc_hunt,p_mean,pet_mean,aridity_index,p_seasonality,frac_snow,high_prec_freq,low_prec_freq,hydroatlas_elev_m,hydroatlas_slope_deg,hydroatlas_temp_mean_c,hydroatlas_precip_mm_yr,hydroatlas_pet_mm_yr,hydroatlas_aridity,hydroatlas_clay_pct,hydroatlas_sand_pct,hydroatlas_forest_pct,hydroatlas_crop_pct,hydroatlas_urban_pct
str,"datetime[μs, UTC]",f64,f64,f64,f64,f64,f64,f64,f64,i64,f32,f32,f64,f32,f32,f32,f32,f32,f32,f32,str,i64,f64,str,f64,i32,i32,f64,f64,f64,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""06923250""",2008-08-03 21:00:00 UTC,37.684306,-92.924639,77.85,78.1,77.1,1.5675,1.57,1.56,4,0.0,32.25,3.908248,0.021026,97496.578125,743.999023,433.099976,0.8589,4237.952148,0.0,"""Niangua River at Windyville, M…",10290110,873.2281,null,370.4886,479,277,3.732098,0.830304,-0.55731,"""sedimentary""","""Red clay, massive clay that is…",3.285525,4.645295,1.413867,0.224341,0.037122,21.939527,252.682059,372.979763,19.974893,129.817771,1079.965809,1195.818559,81.715471,22.698742,34.502088,49.937232,36.317987,0.90795
"""06925250""",2008-08-03 21:00:00 UTC,37.934722,-93.066944,8.07,8.07,8.07,3.85,3.85,3.85,4,0.0,32.209991,3.859223,0.021347,97785.21875,743.775024,434.660004,0.858,4277.632324,0.0,"""Little Niangua River near Mack…",10290110,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""06155030""",2008-08-03 22:00:00 UTC,48.402778,-108.294094,4.525,6.3,3.43,3.0925,3.14,3.06,4,0.0,24.0,0.857963,0.007571,92852.484375,674.590027,344.919983,0.6143,97.024002,0.0,"""Milk River near Dodson MT""",10050004,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""06209500""",2008-08-03 22:00:00 UTC,45.086147,-109.329189,391.5,394.0,389.0,6.185,6.19,6.18,4,0.0,14.959991,1.380905,0.005518,75706.25,659.710022,258.350006,0.5628,16.384001,0.0,"""Rock Creek near Red Lodge MT""",10070006,272.6757,null,2943.08,3796,1948,36.67257,-0.250001,0.968245,"""gneiss""","""Bouldery and sandy colluvium o…",1.963299,3.178244,1.618828,0.072573,0.648379,14.900225,221.105129,2925.833854,158.162552,-18.234882,743.951621,640.952899,94.400685,9.936914,40.575733,60.449562,1.656152,0.0
"""06347000""",2008-08-03 22:00:00 UTC,46.545284,-101.645423,0.0,0.0,0.0,4.1125,4.12,4.11,4,0.0,25.209991,5.350934,0.013481,93343.367188,672.478027,387.630005,0.6265,967.040039,0.0,"""ANTELOPE CREEK NR CARSON, ND""",10130203,617.2218,null,704.7647,846,597,1.948001,0.886806,0.462142,"""sedimentary""","""Shaley or sandy ground; on mix…",1.14426,4.347625,3.799509,0.880889,0.121953,20.296283,300.9579,702.238808,11.886817,55.509135,425.542717,946.016351,44.575564,16.839658,46.5567,2.971704,59.434085,0.0
"""06483500""",2008-08-03 22:00:00 UTC,43.215083,-96.295028,443.0,445.0,441.0,4.665,4.67,4.66,4,0.0,29.51001,3.390811,0.018786,96016.007812,690.878052,407.029999,0.7133,3031.680176,0.0,"""Rock River near Rock Valley, I…",10170204,4133.943,null,468.542,601,373,1.243466,-0.845004,-0.534759,"""sedimentary""","""Wisconsinan loess""",1.927512,4.136433,2.145996,0.723311,0.106453,21.828497,273.688934,466.06119,8.916266,70.780813,682.063204,984.065104,68.35804,20.804621,32.692976,0.0,80.246395,0.0
"""06601000""",2008-08-03 22:00:00 UTC,42.321408,-96.487931,57.125,57.4,56.3,2.4875,2.49,2.48,4,0.0,31.769989,5.019083,0.018864,96586.25,707.742004,409.269989,0.80

In [25]:
# Check for suspicious sentinel values in streamflow and gage height
print("Streamflow stats:")
print(f"  Min: {kept_df['streamflow_cfs_mean'].min()}")
print(f"  Max: {kept_df['streamflow_cfs_mean'].max()}")
print(f"  Rows >= 99999: {kept_df.filter(pl.col('streamflow_cfs_mean') >= 999999).shape[0]}")
print(f"  Rows <= -99999: {kept_df.filter(pl.col('streamflow_cfs_mean') <= -999999).shape[0]}")

print("\nGage height stats:")
print(f"  Min: {kept_df['gage_height_ft_mean'].min()}")
print(f"  Max: {kept_df['gage_height_ft_mean'].max()}")
print(f"  Rows >= 99999: {kept_df.filter(pl.col('gage_height_ft_mean') >= 999999).shape[0]}")
print(f"  Rows <= -99999: {kept_df.filter(pl.col('gage_height_ft_mean') <= -999999).shape[0]}")

print("\nNegative streamflow rows:", kept_df.filter(pl.col("streamflow_cfs_mean") < 0).shape[0])
print("Negative gage height rows:", kept_df.filter(pl.col("gage_height_ft_mean") < 0).shape[0])

Streamflow stats:
  Min: -749998.905
  Max: 197000.0
  Rows >= 99999: 0
  Rows <= -99999: 0

Gage height stats:
  Min: -749999.065
  Max: 73.2825
  Rows >= 99999: 0
  Rows <= -99999: 0

Negative streamflow rows: 247
Negative gage height rows: 108250
